# DS Agent Evaluation — Analysis Notebook

**A rigorous, multi-component evaluation of a tool-use data science agent on the InfiAgent-DABench benchmark.**

This notebook documents the full evaluation pipeline: verifiable scoring, trajectory analysis, failure taxonomy, LLM-as-judge quality scoring, and human validation. Every design decision is motivated and discussed.

---
*Eval framework: `eval/metrics.py` · `eval/trajectory.py` · `eval/taxonomy.py` · `eval/llm_judge.py` · `eval/human_validation.py`*


## Setup

In [ ]:
import os, json
import warnings
from pathlib import Path
from collections import Counter

# Notebooks run from the notebooks/ subdirectory; move up to project root
if Path("data").exists():
    ROOT = Path(".")
elif Path("../data").exists():
    os.chdir("..")
    ROOT = Path(".")
else:
    raise RuntimeError("Could not locate project root — run from project root or notebooks/")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
COLORS = {"sonnet": "#4C72B0", "haiku": "#DD8452"}

FILES = {
    # hard_final agent runs (40 hard tasks, both models)
    "hf_sonnet_run":     "results/hard_final_sonnet_20260626_152647.json",
    "hf_haiku_run":      "results/hard_final_haiku_20260626_153205.json",
    "hf_sonnet_metrics": "results/metrics_hard_final_sonnet_20260626_152647.json",
    "hf_haiku_metrics":  "results/metrics_hard_final_haiku_20260626_153205.json",
    "hf_sonnet_traj":    "results/trajectory_hard_final_sonnet_20260626_152647.json",
    "hf_haiku_traj":     "results/trajectory_hard_final_haiku_20260626_153205.json",
    # judge outputs
    "judge_dev_sonnet":  "results/judge_v3_sonnet.json",
    "judge_dev_haiku":   "results/judge_v3_haiku.json",
    "judge_hf_sonnet":   "results/judge_hard_final_sonnet_20260626_152647.json",
    "judge_hf_haiku":    "results/judge_hard_final_haiku_20260626_153205.json",
    # human validation
    "agreement":         "results/agreement_20260626_145204.json",
    # failure taxonomy
    "taxonomy":          "results/taxonomy.json",
    # judge v1 / v2 / v3 on dev-scale (for rubric evolution table)
    "judge_v1_haiku":    "results/judge_hard_dev_all_haiku_20260622_152459.json",
    "judge_v2_haiku":    "results/judge_v2_haiku.json",
    "judge_v3_haiku":    "results/judge_v3_haiku.json",
    "judge_v1_sonnet":   "results/judge_hard_dev_all_sonnet_combined.json",
    "judge_v2_sonnet":   "results/judge_v2_sonnet.json",
    "judge_v3_sonnet":   "results/judge_v3_sonnet.json",
}

def load(key):
    return json.loads((ROOT / FILES[key]).read_text())

print("Files loaded. Splits:")
print("  Dev-scale: 20 hard tasks (hard_dev_all)")
print("  Hard-final: 40 hard tasks (20 holdout hard + 20 new hard, seed=42)")


## 1 · Evaluation Design

The framework has four independent components, each measuring something different:

| Component | What it measures | Requires ground truth? |
|-----------|-----------------|----------------------|
| **Verifiable eval** | Exact-match correctness of `@var[value]` answers | Yes |
| **Trajectory eval** | Step counts, error rates, failure location | No |
| **Failure taxonomy** | *Why* failures happen — mechanism × source | No |
| **LLM judge** | Analysis quality: OQ (outcome-aware) + TQ (outcome-blind) | Verdict only |

### Key design decisions

**All-or-nothing scoring** — a task either passes all answer variables or fails. Partial credit hides weaknesses and inflates scores on tasks with multiple answer variables.

**Hard-only evaluation** — the full DABench has easy/medium/hard tasks. Hard tasks produce richer failure patterns. Easy tasks produce near-100% pass rates with no discrimination.

**Two-call LLM judge** — Output Quality (OQ) sees the correctness verdict but not the trajectory. Trajectory Quality (TQ) sees the full trajectory but not the verdict or ground truth. This prevents hindsight bias: knowing an answer was wrong would unfairly penalise a sound analytical process.

**Gemini as judge** — using a different model family (Google Gemini Flash) avoids same-family self-preference bias.

**Model comparison controls** — everything held constant except the model: same scaffolding, same skill file, same tasks, same max-iterations, same eval harness.


## 2 · Dataset — InfiAgent-DABench

In [ ]:
data_dir = ROOT / "data/dabench"

questions = [json.loads(l) for l in (data_dir / "questions.jsonl").read_text().splitlines()]
q_df = pd.DataFrame(questions)

print(f"Total tasks:  {len(q_df)}")
print(f"CSV tables:   68")

diff = q_df["level"].value_counts().sort_index()
print(f"\nDifficulty distribution:")
for level, n in diff.items():
    print(f"  {level:6s}: {n}")

# Concept coverage
concepts_flat = []
for cs in q_df["concepts"].dropna():
    if isinstance(cs, list):
        concepts_flat.extend(cs)
    elif isinstance(cs, str):
        concepts_flat.append(cs)
top = Counter(concepts_flat).most_common(8)
print(f"\nTop concepts across {len(set(concepts_flat))} unique types:")
for c, n in top:
    print(f"  {c}: {n}")


In [ ]:
# Task splits
id_to_level = {q["id"]: q["level"] for q in questions}

DEV          = [0,9,18,5,11,62,7,23,28,39, 24,32,55,27,66,69,70,77,118,124]
HARD_DEV_ALL = [7,23,28,39,70,77,118,124,109,144,178,214,282,297,308,574,604,665,685,732]
HARD_HOLDOUT = [137,177,210,224,249,310,378,423,431,521,523,530,590,647,674,722,723,724,734,736]
NEW_HARD     = [30,111,125,220,222,271,273,355,363,376,413,415,424,496,593,662,669,673,725,733]
HARD_FINAL   = sorted(HARD_HOLDOUT + NEW_HARD)

splits = {
    "Dev (20)":          DEV,
    "Hard dev all (20)": HARD_DEV_ALL,
    "Hard holdout (20)": HARD_HOLDOUT,
    "New hard (20)":     NEW_HARD,
    "Hard final (40)":   HARD_FINAL,
}

rows = []
for name, ids in splits.items():
    lvls = Counter(id_to_level.get(i, "?") for i in ids)
    rows.append({"Split": name, "n": len(ids),
                 "easy": lvls.get("easy", 0),
                 "medium": lvls.get("medium", 0),
                 "hard": lvls.get("hard", 0)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nHard-final = 20 holdout hard + 20 new hard tasks (seed=42, never seen during dev)")


## 3 · Verifiable Eval — Pass Rates

In [ ]:
def pass_rate_from_judge(judge_data):
    tasks = [t for t in judge_data["tasks"]
             if t["output_quality"]["score"] is not None]
    passed = sum(1 for t in tasks if t["passed_verifiable"])
    return passed, len(tasks)

def pass_rate_from_metrics(metrics_data):
    tasks = metrics_data["tasks"]
    return sum(1 for t in tasks if t["passed"]), len(tasks)

# Dev-scale (20 hard tasks — partially contaminated, used for development)
dev_j_s = load("judge_dev_sonnet")
dev_j_h = load("judge_dev_haiku")
dev_s_pass, dev_s_n = pass_rate_from_judge(dev_j_s)
dev_h_pass, dev_h_n = pass_rate_from_judge(dev_j_h)

# Hard-final (40 tasks — clean test set: 20 holdout hard + 20 new hard)
hf_m_s = load("hf_sonnet_metrics")
hf_m_h = load("hf_haiku_metrics")
hf_s_pass, hf_s_n = pass_rate_from_metrics(hf_m_s)
hf_h_pass, hf_h_n = pass_rate_from_metrics(hf_m_h)

print(f"{'Split':<30} {'Sonnet 4.6':>15} {'Haiku 4.5':>15}")
print("─" * 62)
for label, sp, sn, hp, hn in [
    ("Dev-scale (20) [contaminated]", dev_s_pass, dev_s_n, dev_h_pass, dev_h_n),
    ("Hard-final (40) [primary]",     hf_s_pass,  hf_s_n,  hf_h_pass,  hf_h_n),
]:
    print(f"{label:<30} {sp}/{sn} ({sp/sn*100:.0f}%){'':<6} "
          f"{hp}/{hn} ({hp/hn*100:.0f}%)")
print()
print("Note: dev-scale and hard-final are disjoint task sets but should not be")
print("pooled — dev tasks were used to tune the skill file (partial contamination).")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Left: pass rates by split — y-axis from 0 ──
splits_lbl = ["Dev-scale\n(20, contaminated)", "Hard-final\n(40, primary test)"]
sonnet_pcts = [dev_s_pass/dev_s_n, hf_s_pass/hf_s_n]
haiku_pcts  = [dev_h_pass/dev_h_n, hf_h_pass/hf_h_n]

x = np.arange(len(splits_lbl)); w = 0.35
ax = axes[0]
b1 = ax.bar(x - w/2, [p*100 for p in sonnet_pcts], w,
            label="Sonnet 4.6", color=COLORS["sonnet"], alpha=0.85)
b2 = ax.bar(x + w/2, [p*100 for p in haiku_pcts],  w,
            label="Haiku 4.5",  color=COLORS["haiku"],  alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(splits_lbl)
ax.set_ylabel("Pass rate (%)"); ax.set_ylim(0, 100)
ax.set_title("Pass Rate: Dev vs Test")
ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=9)

# ── Right: failure source breakdown from taxonomy (hard-final) ──
tax_data = load("taxonomy")
failures = tax_data["failures"]

BUCKETS = {
    "Benchmark\n(underspecified)": ("task_ambiguity",),
    "Agent-induced":               ("constraint_violation", "numeric_mismatch", "output_format_failure"),
    "Scaffolding\n(iter limit)":   ("iteration_limit",),
    "Eval artifact\n(rounding)":   ("rounding_artifact",),
}
SOURCE_COLORS = {
    "Benchmark\n(underspecified)": "#d62728",
    "Agent-induced":               "#4C72B0",
    "Scaffolding\n(iter limit)":   "#7f7f7f",
    "Eval artifact\n(rounding)":   "#bcbd22",
}

bucket_labels = list(BUCKETS.keys())
sonnet_counts = [sum(1 for f in failures if f["model"] == "sonnet" and f["mechanism"] in mechs)
                 for mechs in BUCKETS.values()]
haiku_counts  = [sum(1 for f in failures if f["model"] == "haiku"  and f["mechanism"] in mechs)
                 for mechs in BUCKETS.values()]

x2 = np.arange(len(bucket_labels)); ax2 = axes[1]
ax2.bar(x2 - w/2, sonnet_counts, w, label="Sonnet 4.6", color=COLORS["sonnet"], alpha=0.85)
ax2.bar(x2 + w/2, haiku_counts,  w, label="Haiku 4.5",  color=COLORS["haiku"],  alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels(bucket_labels, fontsize=9)
ax2.set_ylabel("Number of failures")
ax2.set_title("Failure Source — Hard-final (40 tasks)")
ax2.legend()
for bars, counts in [(ax2.patches[:len(bucket_labels)], sonnet_counts),
                     (ax2.patches[len(bucket_labels):], haiku_counts)]:
    for bar, v in zip(bars, counts):
        if v > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, v + 0.05,
                     str(v), ha="center", va="bottom", fontsize=9)

# Annotation: fraction that is benchmark-driven
s_bm_pct = sonnet_counts[0] / sum(sonnet_counts) * 100
h_bm_pct = haiku_counts[0]  / sum(haiku_counts)  * 100
ax2.annotate(f"Benchmark-driven:\n{sonnet_counts[0]}/10 Sonnet ({s_bm_pct:.0f}%)\n"
             f"{haiku_counts[0]}/8 Haiku ({h_bm_pct:.0f}%)",
             xy=(0, 4), xytext=(1.5, 4.2),
             fontsize=8, color="#d62728",
             arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.2))

plt.tight_layout()
plt.savefig("results/fig_pass_rates.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/fig_pass_rates.png")
print(f"\nFailure source breakdown (hard-final):")
for lbl, sc, hc in zip(bucket_labels, sonnet_counts, haiku_counts):
    print(f"  {lbl.replace(chr(10),' '):35s}  Sonnet: {sc}  Haiku: {hc}")


**Observation:** Haiku reversed the dev-scale result on hard-final (80% vs 75%). The gap is narrow and driven by a few swing tasks:

- **Task 424** (Sonnet only — iteration limit): BitConnect prices all <$141; custom thresholds (500/1000) put everything in "Low". Sonnet correctly diagnosed the degenerate case, explored multiple interpretations over 10 steps, and hit the limit. Haiku committed to an ordered list and matched the benchmark's only checked variable.
- **Tasks 363, 722** (Sonnet only): Year truncation and column-selection errors.
- **Tasks 273, 496** (Haiku only): Format and 1% numeric miss.

Neither model dominates across all dimensions — see §8 for the full picture.


## 4 · Trajectory Analysis

In [ ]:
def make_traj_df(traj_data, model):
    rows = []
    for t in traj_data["tasks"]:
        rows.append({
            "task_id":   t["task_id"],
            "model":     model,
            "steps":     t["n_tool_calls"],
            "errors":    t["n_errors"],
            "hit_limit": t.get("hit_max_iterations", False),
        })
    return pd.DataFrame(rows)

hf_s_traj_data = load("hf_sonnet_traj")
hf_h_traj_data = load("hf_haiku_traj")
df_s = make_traj_df(hf_s_traj_data, "Sonnet 4.6")
df_h = make_traj_df(hf_h_traj_data, "Haiku 4.5")
df_all = pd.concat([df_s, df_h], ignore_index=True)

print("Step-count statistics — hard-final (40 tasks each):")
stats = df_all.groupby("model")["steps"].agg(["min", "median", "mean", "max"])
print(stats.to_string())
print()
for m, g in df_all.groupby("model"):
    err = (g["errors"] > 0).mean() * 100
    lim = g["hit_limit"].mean() * 100
    print(f"  {m}: {err:.0f}% tasks with errors  |  {lim:.0f}% hit iteration limit")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
bins = np.arange(1, 13) - 0.5
ax.hist(df_s["steps"], bins=bins, alpha=0.7, color=COLORS["sonnet"],
        label="Sonnet 4.6", density=True)
ax.hist(df_h["steps"], bins=bins, alpha=0.7, color=COLORS["haiku"],
        label="Haiku 4.5",  density=True)
ax.axvline(df_s["steps"].median(), color=COLORS["sonnet"], linestyle="--",
           linewidth=1.8, label=f"Sonnet median={df_s['steps'].median():.0f}")
ax.axvline(df_h["steps"].median(), color=COLORS["haiku"],  linestyle="--",
           linewidth=1.8, label=f"Haiku median={df_h['steps'].median():.0f}")
ax.set_xlabel("Steps (tool calls)"); ax.set_ylabel("Density")
ax.set_title("Step-count Distribution — Hard-final")
ax.legend(fontsize=8)

def task_costs(run_key):
    return [r["cost_usd"] for r in load(run_key)["results"]]

costs_s = task_costs("hf_sonnet_run")
costs_h = task_costs("hf_haiku_run")

ax2 = axes[1]
ax2.boxplot([costs_s, costs_h], labels=["Sonnet 4.6", "Haiku 4.5"],
            patch_artist=True,
            boxprops=dict(facecolor="white"),
            medianprops=dict(color="black", linewidth=2))
rng = np.random.default_rng(42)
for i, (costs, color) in enumerate([(costs_s, COLORS["sonnet"]),
                                    (costs_h, COLORS["haiku"])], 1):
    jitter = rng.uniform(-0.08, 0.08, len(costs))
    ax2.scatter(np.full(len(costs), i) + jitter, costs,
                alpha=0.4, color=color, s=18, zorder=3)
ax2.set_ylabel("Cost per task (USD)")
ax2.set_title("Cost Distribution — Hard-final")
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.3f"))

plt.tight_layout()
plt.savefig("results/fig_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()

total_s = sum(costs_s); total_h = sum(costs_h)
print(f"Total cost:   Sonnet ${total_s:.2f}  |  Haiku ${total_h:.2f}")
print(f"Cost/task:    Sonnet ${total_s/40:.3f}  |  Haiku ${total_h/40:.3f}")
print(f"\nHaiku costs {total_h/total_s:.1f}x more despite cheaper per-token rates —")
print("it takes more steps per task, generating more total tokens.")


## 5 · Failure Taxonomy

In [ ]:
tax = load("taxonomy")
failures = tax["failures"]

fail_df = pd.DataFrame([{
    "task_id":   f["task_id"],
    "model":     f["model"],
    "mechanism": f["mechanism"],
    "source":    f["source"],
    "shared":    f.get("cross_model_fail", False),
} for f in failures])

# Mechanism × model counts
mech_pivot = (fail_df.groupby(["mechanism", "source", "model"])["task_id"]
              .count().unstack(fill_value=0).reset_index())
mech_pivot.columns.name = None
print("Failure taxonomy — hard-final (40 tasks per model):")
print(mech_pivot.to_string(index=False))
print()

sonnet_fail = set(fail_df[fail_df["model"] == "sonnet"]["task_id"])
haiku_fail  = set(fail_df[fail_df["model"] == "haiku"]["task_id"])
shared      = sonnet_fail & haiku_fail
print(f"Shared failures (both models): {len(shared)} — tasks {sorted(shared)}")
print(f"Sonnet only:                   {len(sonnet_fail - haiku_fail)}")
print(f"Haiku only:                    {len(haiku_fail  - sonnet_fail)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Mechanism breakdown
all_mechs = fail_df["mechanism"].unique()
y_pos = np.arange(len(all_mechs)); w = 0.35
ax = axes[0]
for i, (model, color, lbl) in enumerate([
    ("sonnet", COLORS["sonnet"], "Sonnet 4.6"),
    ("haiku",  COLORS["haiku"],  "Haiku 4.5"),
]):
    sub = (fail_df[fail_df["model"] == model]
           .groupby("mechanism")["task_id"].count())
    vals = [sub.get(m, 0) for m in all_mechs]
    ax.barh(y_pos + (i - 0.5) * w, vals, w, label=lbl, color=color, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([m.replace("_", "\n") for m in all_mechs], fontsize=9)
ax.set_xlabel("Number of failures")
ax.set_title("Failure Mechanisms — Hard-final")
ax.legend()

# Shared vs model-specific
cats = ["Shared\n(both)", "Sonnet only", "Haiku only"]
vals = [len(shared), len(sonnet_fail - haiku_fail), len(haiku_fail - sonnet_fail)]
clrs = ["#7F7F7F", COLORS["sonnet"], COLORS["haiku"]]
ax2 = axes[1]
bars = ax2.bar(cats, vals, color=clrs, width=0.5, alpha=0.85)
ax2.set_ylabel("Number of tasks")
ax2.set_title("Shared vs Model-specific Failures")
ax2.set_ylim(0, max(vals) + 2)
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.1,
             str(v), ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("results/fig_taxonomy.png", dpi=150, bbox_inches="tight")
plt.show()


**Key taxonomy findings:**

- **`task_ambiguity` dominates (4 tasks each).** These are underspecification failures — the question leaves implementation details underconstrained (random seed, split method, output ordering), so both models made reasonable choices that don't match the benchmark's specific expected values. Tasks 137, 224, 521: ML model metrics where both models land within 1–2% of GT but fail exact match. Task 734: multi-continent correlation analysis where the question doesn't specify output ordering; the model computed 0.96 (matching one of five GT values) but not the others. A meaningful fraction of "failures" are not agent failures.

- **`iteration_limit` (1 Sonnet, 0 Haiku).** Task 424: Sonnet correctly diagnosed the degenerate dataset but ran out of steps. A principled failure, not a reasoning failure — but the completion gate capped TQ at 1 (no `@var[]` answer produced).

- **Shared failures** are all `task_ambiguity` — both models reached the same wrong answer by following correct reasoning against underspecified tasks.

- **Zero unrecovered code errors.** Every exception was caught and handled within the trajectory.


## 6 · LLM Judge — Quality Beyond Correctness

Verifiable eval is binary. It misses quality variation among correct answers and gives no credit for sound processes on benchmark-error tasks. The LLM judge adds two continuous dimensions (0–3):

| Dimension | What it measures | Information provided |
|-----------|-----------------|---------------------|
| **Output Quality (OQ)** | Depth, rigour, interpretation of the final answer | Question + verdict. Not trajectory or raw GT. |
| **Trajectory Quality (TQ)** | Process quality: efficiency, error handling, approach | Question + full trajectory. Not verdict or GT. |

### TQ rubric evolution — why three iterations were required


In [ ]:
def tq_stats(judge_data):
    scores = [t["trajectory_quality"]["score"] for t in judge_data["tasks"]
              if t["trajectory_quality"]["score"] is not None]
    return round(np.mean(scores), 2), sum(1 for s in scores if s == 3)

evo_rows = []
for version, h_key, s_key, change in [
    ("v1 (baseline)",
     "judge_v1_haiku", "judge_v1_sonnet",
     "Baseline rubric — four criteria, no gates"),
    ("v2 (+ completion gate)",
     "judge_v2_haiku", "judge_v2_sonnet",
     "No @var[] answer → score capped at 1"),
    ("v3 (+ error/redundancy gate)",
     "judge_v3_haiku", "judge_v3_sonnet",
     "Traceback/re-run → cap at 2; CSV reload every step → cap at 2"),
]:
    h_mean, h_3 = tq_stats(load(h_key))
    s_mean, s_3 = tq_stats(load(s_key))
    evo_rows.append({
        "Version": version, "Key change": change,
        "Haiku mean": h_mean, "Haiku TQ=3": h_3,
        "Sonnet mean": s_mean, "Sonnet TQ=3": s_3,
    })

print(pd.DataFrame(evo_rows).to_string(index=False))
print()
print("v2 regressed on Haiku (TQ=3 went 19→20): completion gate alone did not")
print("penalise pervasive CSV reloading (one reload per step in every task).")
print("v3 added an explicit redundancy gate; TQ=3 is now selective.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# TQ mean by rubric version
versions_lbl = ["v1\n(baseline)", "v2\n(+ completion\ngate)", "v3\n(+ redundancy\ngate)"]
haiku_means  = [2.95, 3.00, 1.95]
sonnet_means = [2.95, 2.90, 2.05]
x = np.arange(3); w = 0.35
ax = axes[0]
ax.bar(x - w/2, haiku_means,  w, label="Haiku 4.5",  color=COLORS["haiku"],  alpha=0.85)
ax.bar(x + w/2, sonnet_means, w, label="Sonnet 4.6", color=COLORS["sonnet"], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(versions_lbl)
ax.set_ylabel("Mean TQ score (0–3)"); ax.set_ylim(0, 3.5)
ax.set_title("TQ Rubric Evolution — Dev-scale (20 tasks)")
ax.axhline(3, color="grey", linestyle=":", linewidth=1)
ax.legend()
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

# v3 TQ distribution on hard-final
j_hf_s = load("judge_hf_sonnet")
j_hf_h = load("judge_hf_haiku")

def tq_dist(j):
    sc = [t["trajectory_quality"]["score"] for t in j["tasks"]
          if t["trajectory_quality"]["score"] is not None]
    return [sc.count(s) for s in [0, 1, 2, 3]]

x2 = np.arange(4); ax2 = axes[1]
ax2.bar(x2 - w/2, tq_dist(j_hf_s), w, label="Sonnet 4.6", color=COLORS["sonnet"], alpha=0.85)
ax2.bar(x2 + w/2, tq_dist(j_hf_h), w, label="Haiku 4.5",  color=COLORS["haiku"],  alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(["0", "1", "2", "3"])
ax2.set_xlabel("TQ score"); ax2.set_ylabel("Number of tasks")
ax2.set_title("TQ Distribution — Hard-final (v3 rubric)")
ax2.legend()

plt.tight_layout()
plt.savefig("results/fig_tq_evolution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Two-call design validation: TQ(failed) vs TQ(passed)
def pass_fail_split(judge_data):
    rows = [(t["passed_verifiable"],
             t["output_quality"]["score"],
             t["trajectory_quality"]["score"])
            for t in judge_data["tasks"]
            if t["output_quality"]["score"] is not None
            and t["trajectory_quality"]["score"] is not None]
    passed = [(oq, tq) for v, oq, tq in rows if v]
    failed = [(oq, tq) for v, oq, tq in rows if not v]
    return passed, failed

s_pass, s_fail = pass_fail_split(j_hf_s)
h_pass, h_fail = pass_fail_split(j_hf_h)

print("Two-call design validation — TQ(failed) vs TQ(passed):")
print(f"\n{'':28} {'Sonnet 4.6':>12} {'Haiku 4.5':>12}")
print(f"{'TQ — passed tasks':28} {np.mean([t for _, t in s_pass]):>12.2f} "
      f"{np.mean([t for _, t in h_pass]):>12.2f}")
print(f"{'TQ — failed tasks':28} {np.mean([t for _, t in s_fail]):>12.2f} "
      f"{np.mean([t for _, t in h_fail]):>12.2f}")
print(f"{'OQ — passed tasks':28} {np.mean([o for o, _ in s_pass]):>12.2f} "
      f"{np.mean([o for o, _ in h_pass]):>12.2f}")
print(f"{'OQ — failed tasks':28} {np.mean([o for o, _ in s_fail]):>12.2f} "
      f"{np.mean([o for o, _ in h_fail]):>12.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, p_vals, f_vals, title in [
    (axes[0],
     [np.mean([t for _, t in s_pass]), np.mean([t for _, t in h_pass])],
     [np.mean([t for _, t in s_fail]), np.mean([t for _, t in h_fail])],
     "Trajectory Quality — passed vs failed"),
    (axes[1],
     [np.mean([o for o, _ in s_pass]), np.mean([o for o, _ in h_pass])],
     [np.mean([o for o, _ in s_fail]), np.mean([o for o, _ in h_fail])],
     "Output Quality — passed vs failed"),
]:
    x = np.arange(2); w = 0.35
    ax.bar(x - w/2, p_vals, w, label="Passed tasks", color="#2ca02c", alpha=0.85)
    ax.bar(x + w/2, f_vals, w, label="Failed tasks", color="#d62728", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(["Sonnet 4.6", "Haiku 4.5"])
    ax.set_ylim(0, 3.4); ax.set_ylabel("Mean score (0–3)")
    ax.set_title(title); ax.legend()
    ax.axhline(2, color="grey", linestyle=":", linewidth=1, alpha=0.6)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("results/fig_judge_pass_fail.png", dpi=150, bbox_inches="tight")
plt.show()


**Headline finding: TQ(failed) ≈ or ≥ TQ(passed).** The outcome-blind TQ judge correctly identifies that failed tasks often involved sound analytical processes — failures were driven by underspecification and degenerate datasets, not poor reasoning. Holds clearly for Sonnet (TQ failed 2.30 vs passed 2.13); Haiku essentially equal (2.00 vs 2.03). This is the two-call design working as intended.

**OQ(failed) is appropriately low** (wrong answers score low). The large OQ gap between passed and failed tasks, combined with the small or reversed TQ gap, is exactly the discrimination pattern we designed for.


### Case study — Task 424: where the metric, and even the eval, struggle

**Setup:** The BitConnect dataset has all prices below $141. The task asks for feature importances after encoding price range as Low/Medium/High using thresholds of 500 and 1000. Because every row falls in "Low," the encoded column is constant — feature importances are all zero and the task as specified has no meaningful answer.

**What happened:**

| | Sonnet 4.6 | Haiku 4.5 |
|---|---|---|
| Verifiable result | **FAIL** | **PASS** |
| What the model did | Correctly diagnosed the degenerate dataset; explored multiple re-interpretations over 10 steps; refused to output a meaningless result; hit the iteration limit | Committed to a format-valid ordered list `[Open, High, Low]`; the benchmark only checked `@feature2=High`; passed by coincidence |
| OQ score | 0 | 3 |
| TQ score | **1** | **2** |

**The tension this surfaces:**

Verifiable correctness actively rewarded the worse model. The outcome-blind TQ judge was supposed to correct this — to credit Sonnet's more principled process. It didn't. Sonnet's trajectory scored *below* Haiku's (TQ 1 vs 2) because the completion gate caps any run that produces no `@var[]` answer at 1, regardless of why it didn't terminate.

This exposes a genuine tension in the rubric design: the completion gate was added to stop rewarding clean-looking flailing (v2 had near-ceiling TQ because models that flailed elegantly still got 3s). But the gate can't distinguish *"didn't finish because it flailed"* from *"didn't finish because it correctly determined no valid answer exists."* Completion is an imperfect process proxy — sometimes not-completing is the right behavior.

**The fix this points to:** the agent needs a graceful-termination affordance — a way to signal principled non-completion (e.g., `@answer[INDETERMINATE: dataset is degenerate because...]`) that the rubric can credit separately from failure-to-complete. Without it, the judge has no signal to distinguish the two cases and must apply the gate uniformly.

**What this demonstrates:** Task 424 shows both the value and the limits of multi-dimensional evaluation. Correctness alone was actively misleading. The process dimension (TQ) partially rescued the story — the judge's OQ=0 correctly flagged that Sonnet produced nothing useful — but TQ didn't fully rescue it because the rubric lacked the affordance to credit principled non-completion. The case is a concrete design input for the next iteration of both the agent and the rubric.


## 7 · Human Validation — Judge Agreement

In [ ]:
agr = load("agreement")

print("Human validation — 12-task stratified batch (v3 rubric)")
print("Task selection: contested, error recovery, redundancy, TQ=3 exemplars")
print()

for key, label in [("output_quality", "OUTPUT QUALITY"),
                   ("trajectory_quality", "TRAJECTORY QUALITY")]:
    s = agr[key]
    print(f"{label}  (n={s['n']})")
    print(f"  Weighted kappa (linear):      {s['linear_kappa']}")
    print(f"  Krippendorff alpha (ordinal): {s['ordinal_alpha']}")
    print(f"  Exact agreement:              {s['exact_agreement_pct']}%")
    print(f"  Within-1 agreement:           {s['within1_pct']}%")
    print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, key, title in [
    (axes[0], "output_quality",    "Output Quality — Score Distribution (n=12)"),
    (axes[1], "trajectory_quality","Trajectory Quality — Score Distribution (n=12)"),
]:
    s = agr[key]
    scores = [0, 1, 2, 3]
    human = [s["score_dist_human"].get(str(sc), 0) for sc in scores]
    judge = [s["score_dist_judge"].get(str(sc), 0) for sc in scores]
    x = np.arange(4); w = 0.35
    ax.bar(x - w/2, human, w, label="Human",  color="#4e9a65", alpha=0.85)
    ax.bar(x + w/2, judge, w, label="Judge",  color="#9b59b6", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(scores)
    ax.set_xlabel("Score (0–3)"); ax.set_ylabel("Count")
    ax.set_title(title); ax.legend()

plt.tight_layout()
plt.savefig("results/fig_human_validation.png", dpi=150, bbox_inches="tight")
plt.show()


**Calibration findings:**

**OQ (kappa = 0.59, "substantial"):** Agreement is solid at extremes (0/1 align well). The main gap is the 2→3 boundary: the judge scored 7 tasks OQ=3, the human scored only 2. The judge is systematically generous about what counts as "validates assumptions + interprets result." One large outlier: task 178 Sonnet (human=1, judge=3) — human required explicit numeric interpretation; judge credited data inspection and format validation.

**TQ (kappa = 0.33, "fair"; within-1 = 100%):** No catastrophic disagreements — both rater and judge agree on the tier. The human gave 3 TQ=3s vs judge's 1: two cases where the human did not flag a minor CSV reload the judge penalised under the redundancy gate. The completion gate created one H=2/J=1 disagreement: human credited the sound process; the rubric caps at 1 when no `@var[]` answer was produced.

Both gaps are **explainable and directional** (not random noise). This is the desired outcome of human validation — it tells us precisely where to tighten the rubric in the next iteration.


## 8 · Cross-model Comparison — Full Summary

In [ ]:
hf_s_run_data  = load("hf_sonnet_run")
hf_h_run_data  = load("hf_haiku_run")
hf_s_traj_data = load("hf_sonnet_traj")
hf_h_traj_data = load("hf_haiku_traj")

def full_summary(run_data, metrics_data, traj_data, judge_data, label):
    n      = len(run_data["results"])
    costs  = [r["cost_usd"] for r in run_data["results"]]
    passed = sum(1 for t in metrics_data["tasks"] if t["passed"])
    steps  = [t["n_tool_calls"]       for t in traj_data["tasks"]]
    errors = sum(1 for t in traj_data["tasks"] if t["n_errors"] > 0)
    lim    = sum(1 for t in traj_data["tasks"] if t.get("hit_max_iterations"))
    oq     = [t["output_quality"]["score"]     for t in judge_data["tasks"]
              if t["output_quality"]["score"]    is not None]
    tq     = [t["trajectory_quality"]["score"] for t in judge_data["tasks"]
              if t["trajectory_quality"]["score"] is not None]
    return {
        "Model":              label,
        "Tasks (n)":          n,
        "Pass rate":          f"{passed}/{n} ({passed/n*100:.0f}%)",
        "Total cost":         f"${sum(costs):.2f}",
        "Cost per task":      f"${sum(costs)/n:.3f}",
        "Median steps":       int(np.median(steps)),
        "Mean steps":         round(np.mean(steps), 1),
        "Tasks w/ errors":    f"{errors} ({errors/n*100:.0f}%)",
        "Iteration limit hits": lim,
        "Mean OQ (0–3)":      round(np.mean(oq), 2),
        "Mean TQ (0–3)":      round(np.mean(tq), 2),
        "TQ=3 (exemplary)":   sum(1 for s in tq if s == 3),
        "TQ=1 (flawed)":      sum(1 for s in tq if s == 1),
    }

s_sum = full_summary(hf_s_run_data, hf_m_s, hf_s_traj_data, j_hf_s, "Sonnet 4.6")
h_sum = full_summary(hf_h_run_data, hf_m_h, hf_h_traj_data, j_hf_h, "Haiku 4.5")

cmp = pd.DataFrame([s_sum, h_sum]).set_index("Model").T
print("Hard-final comparison (40 hard tasks):")
print(cmp.to_string())


In [ ]:
# Radar chart: multi-dimensional comparison
hf_s_traj_tasks = hf_s_traj_data["tasks"]
hf_h_traj_tasks = hf_h_traj_data["tasks"]

def model_radars(run_key, metrics_data, traj_tasks, judge_data):
    costs  = [r["cost_usd"] for r in load(run_key)["results"]]
    passed = sum(1 for t in metrics_data["tasks"] if t["passed"])
    n      = len(costs)
    steps  = [t["n_tool_calls"] for t in traj_tasks]
    err_free = 1 - sum(1 for t in traj_tasks if t["n_errors"] > 0) / n
    oq = np.mean([t["output_quality"]["score"]     for t in judge_data["tasks"]
                  if t["output_quality"]["score"]    is not None]) / 3
    tq = np.mean([t["trajectory_quality"]["score"] for t in judge_data["tasks"]
                  if t["trajectory_quality"]["score"] is not None]) / 3
    return {
        "pass_rate": passed / n,
        "cost": sum(costs),
        "steps": np.mean(steps),
        "err_free": err_free,
        "oq": oq,
        "tq": tq,
    }

s_r = model_radars("hf_sonnet_run", hf_m_s, hf_s_traj_tasks, j_hf_s)
h_r = model_radars("hf_haiku_run",  hf_m_h, hf_h_traj_tasks, j_hf_h)

max_cost  = max(s_r["cost"],  h_r["cost"])
max_steps = max(s_r["steps"], h_r["steps"])

dims_lbl = ["Pass\nrate", "Cost\nefficiency", "Step\nefficiency",
            "Error-free\nrate", "Output\nquality", "Trajectory\nquality"]

def to_radar(r):
    return [r["pass_rate"],
            1 - r["cost"]  / max_cost,
            1 - r["steps"] / max_steps,
            r["err_free"],
            r["oq"],
            r["tq"]]

s_vals = to_radar(s_r); h_vals = to_radar(h_r)
N = len(dims_lbl)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
s_plot = s_vals + s_vals[:1]; h_plot = h_vals + h_vals[:1]; a_plot = angles + angles[:1]

fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))
ax.plot(a_plot, s_plot, color=COLORS["sonnet"], linewidth=2, label="Sonnet 4.6")
ax.fill(a_plot, s_plot, color=COLORS["sonnet"], alpha=0.15)
ax.plot(a_plot, h_plot, color=COLORS["haiku"],  linewidth=2, label="Haiku 4.5")
ax.fill(a_plot, h_plot, color=COLORS["haiku"],  alpha=0.15)
ax.set_thetagrids(np.degrees(angles), dims_lbl, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=7)
ax.set_title("Multi-dimensional Model Comparison\n(hard-final, 40 tasks)", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig("results/fig_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Cost/step efficiency normalised relative to the less efficient model.")


## 9 · Conclusions

### What the data shows

| Dimension | Sonnet 4.6 | Haiku 4.5 | Assessment |
|-----------|-----------|----------|-----------|
| Pass rate — dev-scale (20)¹ | 85% | 80% | Sonnet led on dev |
| **Pass rate — hard-final (40)²** | **75%** | **80%** | **Haiku marginally ahead on test** |
| Total cost (40 tasks) | $2.19 | $3.28 | Sonnet 33% cheaper |
| Median steps per task | 2 | 3 | Sonnet more efficient |
| Code errors (% tasks) | 0% | 5% | Sonnet cleaner |
| Mean TQ (process quality) | 2.17 | 2.02 | Sonnet cleaner trajectories |
| Mean OQ (answer quality) | 2.42 | 2.55 | Comparable — within judge noise (kappa=0.59; known 2→3 generosity) |

¹ Dev-scale: 20 hard tasks from the development set. Partially contaminated — the skill file was tuned on some of these tasks' failure modes.  
² Hard-final: the primary result. 20 hard holdout + 20 new hard tasks (seed=42), never seen during development.

**Sonnet is cheaper, faster, and produces cleaner trajectories. Pass rates are near parity on the clean test set; output quality is comparable across tiers — the 0.13 OQ gap is smaller than the judge's characterized noise and should not be over-read.**

### Why this eval framework matters

**1. Verifiable eval alone overstates agent failure.** 40–50% of failures are underspecification or ambiguity (task_ambiguity): 4/10 Sonnet failures, 4/8 Haiku failures. Without the taxonomy and judge, these are indistinguishable from genuine agent failures.

**2. The two-call judge design works.** TQ(failed) ≈ or ≥ TQ(passed) — the outcome-blind judge correctly credits sound processes even when the ground truth is wrong. Holds clearly for Sonnet (2.30 vs 2.13); Haiku essentially equal (2.00 vs 2.03). A single-call design (both dimensions seeing the verdict) would have hindsight-penalised these processes.

**3. Rubric iteration was necessary and informative.** Three rubric versions were required to achieve useful TQ spread. Each regression revealed something real about model behavior: v1 couldn't distinguish Sonnet from Haiku; v2 surfaced Haiku's completion failures; v3 surfaced Haiku's pervasive redundancy pattern. The rubric evolution is a finding, not just a process artifact.

**4. Human validation reveals systematic bias, not noise.** Judge generous on OQ (2→3 boundary), strict on TQ (redundancy gate). Both are actionable: tighten OQ rubric, add worked examples for redundancy. Kappa 0.59 / 0.33 with 100% within-1 TQ agreement is a reasonable outcome for a 0–3 scale on complex analytical trajectories.

**5. Hard-only evaluation amplifies the failure story.** Easy tasks hit >95% pass rates with minimal discrimination. Hard tasks surface the mechanistically interesting failures.

### Honest caveats

- The skill file was tuned on Sonnet's dev-set failure modes — a structural advantage for Sonnet that can't be fully controlled for.
- N=40 for hard-final means 3–4 swing tasks can flip the ranking. The ranking reversal from dev to test (Sonnet led 85/80, then trailed 75/80) is within this noise range.
- Judge runs used Gemini Flash free-tier (5 RPM, 15s delays). A paid account would allow zero-delay, higher-throughput runs.
